# 06 — Ad-hoc country → admin1 summary

Summarizes one country's admin2 hazard-exposure results up to **admin1**, across **all hazards**, for prioritizing an admin1 unit. Because global/regional normalization is meaningless within a single country, only **country-scope** indices/classes (`cindex`, `cexpi_*`, `cexpc_*`) are computed — normalized across that country's admin1 units.

## Inputs

- Global per-hazard admin2 CSVs from notebook 02: `{SOURCE_FOLDER}/{hazard}_{date}.csv`
- Optional weight table from notebook 00: `{COUNTS_TABLE}` (`adm2_ucode, hazard_code, hazs_count`). If absent, `hazs_mean`/`std` fall back to a population weight.
- `config/hazard_info.json` (for hazard short codes used as wide-column prefixes)

## Outputs

- **Long**: `{OUTPUT_FOLDER}/results/{COUNTRY}/adm1_summary/{COUNTRY}_admin1_long_{date}.csv` — one row per admin1 × hazard.
- **Wide**: `{OUTPUT_FOLDER}/results/{COUNTRY}/adm1_summary/{COUNTRY}_admin1_wide_hazard_exposure_{date}.xlsx` and `{COUNTRY}_admin1_exposure_{date}.xlsx` — one row per admin1, per-hazard column blocks plus cross-hazard prioritization columns, colour-coded. (No wide CSV is written.)

## Aggregation model

Counts (`pop_*`, `aexp/amexp/afexp/u18_*`) are summed across admin2. `hazs_min/max` roll up via min/max; `hazs_mean/std` are count-weighted pooled statistics; `hazs_median` is **dropped** (non-aggregable). All log/relative inputs and every index/class are **recomputed** from the summed counts — never averaged.

### Population is hazard-independent

Population is a property of the **place**, not of the hazard. Every hazard must therefore report an identical population for a given unit.

The per-hazard tables cannot supply this on their own: they blank population for units flagged `Insufficient Data` / `Not Applicable`, and they carry `No Data` units (the hazard raster does not cover them) with null hazard statistics. Because each hazard flags a *different* set of units, reading population from them makes the denominator drift per hazard.

Two rules keep it stable:

| Row type | Population | Exposure | Hazard stats |
|---|---|---|---|
| Covered | counted | as measured | pooled |
| `No Data` (no raster coverage) | **counted** | **0** | excluded |
| `Insufficient Data` / `Not Applicable` | **counted** (from reference) | blank | excluded |
| Country in `POP_BLANK_COUNTRIES` | — | — | no row at all |

Population is sourced from `POP_REFERENCE` (`admin2_population.csv`), independently of any hazard table — the same approach notebook 05 uses to keep the SDMX population stream free of hazard×admin2 status collisions.

> **Regression (2026-09).** A single `df[df.status == ""]` filter in `aggregate_counts` dropped flagged rows from the population sums as well as the hazard pooling. This shrank the denominator per hazard — Egypt river flood reported 107,276,368 instead of 117,477,477, and Cairo's relative exposure read **15.61% instead of 8.96%** — removed 16.2 billion person-rows globally, and made 821 admin1 units vanish from the river-flood output entirely. The exposed counts were never wrong; only the denominator was. Guarded by `validate_population_totals` (which halts the run) and `tests/test_aggregation.py`.

### Coverage columns

Hazard statistics are only meaningful where the raster actually covers the unit, so each row carries how much of it was seen:

| Column | Meaning |
|---|---|
| `n_admin2_covered` | admin2 children that have hazard data |
| `n_admin2` | admin2 children counted in total |

Read them as a pair — "21 of 47". They are written **adjacent** in every output, because `n_admin2_covered` on its own is meaningless without its total. They are two numeric columns rather than one `21/47` string on purpose: Excel silently autoconverts such text to a date on open (`5/12` becomes `05-Dec`), corrupting it in every workbook.

Where `n_admin2_covered` is well below `n_admin2`, the hazard statistics (`hazs_mean`/`std`/`min`/`max`) rest on only part of the unit and should be read with care. Population and exposure are unaffected; they are always complete.

**No area-share percentage is published**, deliberately. A "% of land covered" figure cannot be read the same way for two hazards: `river_flood_100yr` covers ~1.8% of a typical admin2 because a 100-year floodplain *is* a small part of a region, while heatwave covers ~100%. Printing 1.8% beside 100% invites a reader to dismiss a perfectly sound flood statistic as "poorly covered". A unit either has hazard data or it does not, and that is unambiguous for every hazard.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import os
import pandas as pd
import numpy as np

import aggregation_functions as agg
from indicator_functions import IndicatorUtils

iu = IndicatorUtils()
pd.set_option("display.max_columns", 500)

In [ ]:
# ============================================================
# Parameters (edit these)
# ============================================================
# List of country ucodes to summarize. Set to None to process every country
# found in the source CSVs.
# COUNTRY_UCODES = ["EGY_V1"]
COUNTRY_UCODES = None   # all countries
PROD_DATE = "2026-09-15" # YYYY-MM-DD
ADM2_PROD_DATE = "2026-08-31" # YYYY-MM-DD
SOURCE_FOLDER = r"C:\Users\anjpu\Desktop\UNICEF\repo\CCRI\new_scripts\output_mali\global\CSV"
OUTPUT_FOLDER = r"C:\Users\anjpu\Desktop\UNICEF\repo\CCRI\new_scripts\output_mali"
COUNTS_TABLE = f"output/aux/hazs_count_2026-09-11.csv"  # optional; None to force pop weight
# Independent admin2 population source. Population is a property of the PLACE,
# not of the hazard: the per-hazard tables blank it for suppressed units, so
# reading it from them makes the denominator drift per hazard. Sourcing it here
# is what keeps every hazard reporting identical totals (same approach NB05
# already uses for the SDMX population stream).
POP_REFERENCE = "output/aux/admin2_population.csv"

POP = agg.POPULATION_CLASSES

# Which per-hazard values to carry into the WIDE prioritization file
# Children-focused hazard_exposure wide view: exposure class + index use the
# under-18 columns; cindex (hazard magnitude) is population-independent.
WIDE_VALUES = ["cexpi_under_18_total", "cexpc_under_18_total", "cindex"]


hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
short_code = {r["code"]: r["short_code"] for _, r in hazard_info.iterrows()}
counts_df = agg.load_counts_table(COUNTS_TABLE)
print("weight table:", "loaded" if counts_df is not None else "MISSING -> pop_total fallback")

pop_ref = pd.read_csv(POP_REFERENCE) if os.path.exists(POP_REFERENCE) else None
print("population reference:", f"loaded ({len(pop_ref):,} admin2)" if pop_ref is not None
      else "MISSING -> population will be read from the hazard tables (drifts per hazard!)")


# Discover hazard CSVs in the source folder, then keep only hazards that are
# still defined in hazard_info.json. This drops retired hazards (e.g. the old
# non-grid complex_emergencies_acled) even if a stale CSV lingers in the folder.
valid_codes = set(hazard_info["code"])
discovered = agg.discover_hazard_codes(SOURCE_FOLDER, ADM2_PROD_DATE)
codes = [c for c in discovered if c in valid_codes]
dropped = [c for c in discovered if c not in valid_codes]
if dropped:
    print(f"skipping {len(dropped)} hazard(s) not in hazard_info.json: {dropped}")
print(f"hazards: {len(codes)}")


# ============================================================
# Which countries can actually be summarized to admin1?
# ============================================================
# Two structural cases are excluded:
#  * 1:1 admin1->admin2 -- aggregation is a pass-through, so the admin1 summary
#    would just restate the admin2 data at the same granularity.
#  * a single admin1 unit -- there are no peers to normalize against; these are
#    also a subset of the 1:1 case whenever the country has one admin2.
_probe = agg.load_hazard_global(codes[0], ADM2_PROD_DATE, SOURCE_FOLDER)
_structure = (_probe.groupby("adm0_ucode")
                    .agg(n_adm1=("adm1_ucode", "nunique"),
                         n_adm2=("adm2_ucode", "nunique")))

_requested = COUNTRY_UCODES if COUNTRY_UCODES is not None else sorted(_structure.index)
_missing = [u for u in _requested if u not in _structure.index]

one_to_one = [u for u in _requested if u in _structure.index
              and _structure.loc[u, "n_adm1"] == _structure.loc[u, "n_adm2"]]
single_adm1 = [u for u in _requested if u in _structure.index
               and _structure.loc[u, "n_adm1"] == 1]
# single_adm1 with >1 admin2 is not 1:1, but still cannot be classified.
SKIP = sorted(set(one_to_one) | set(single_adm1))
COUNTRIES_TO_RUN = [u for u in _requested if u not in SKIP and u in _structure.index]

print(f"requested: {len(_requested)}")
if _missing:
    print(f"  not in source data ({len(_missing)}): {_missing[:10]}")
print(f"  skipped 1:1 admin1==admin2 : {len(one_to_one)}")
print(f"  skipped single admin1 unit : {len(single_adm1)}")
print(f"  -> summarizing {len(COUNTRIES_TO_RUN)} country(ies)")

In [ ]:
# ============================================================
# Main loop: one country at a time
# ============================================================
# Cells 3-8 of the original single-country notebook are combined here so the
# whole pipeline (aggregate -> long -> wide -> two Excel views -> verify) runs
# per country. The per-country logic inside the loop is unchanged.
summary_rows = []

for _n, COUNTRY_UCODE in enumerate(COUNTRIES_TO_RUN, 1):
    print(f'\n{"============================================================"}')
    print(f'[{_n}/{len(COUNTRIES_TO_RUN)}] {COUNTRY_UCODE}')

    out_dir = f'{OUTPUT_FOLDER}/results/{COUNTRY_UCODE}/adm1_summary'
    os.makedirs(out_dir, exist_ok=True)

    try:
        # ============================================================
        # Per-hazard: filter country -> aggregate to admin1 -> country-scope indices
        # ============================================================
        long_rows = []
        for code in codes:
            df = agg.load_hazard_global(code, ADM2_PROD_DATE, SOURCE_FOLDER)
            c = df[df["adm0_ucode"] == COUNTRY_UCODE].copy()
            if c.empty:
                print(f"  {code}: no rows for {COUNTRY_UCODE}, skipped")
                continue

            # Population from the independent reference BEFORE aggregating, so
            # suppressed units still contribute their people to the denominator.
            if pop_ref is not None:
                c = agg.apply_population_reference(c, pop_ref)
            c = agg.weight_table_join(c, counts_df, code)
            c = agg.recompute_exposure_inputs(c)
            a1 = agg.aggregate_counts(c, "adm1_ucode")
            if a1.empty:
                print(f"  {code}: no usable admin1 units (all flagged), skipped")
                continue
            a1 = agg.recompute_exposure_inputs(a1)

            # Country-scope indices/classes, normalized across this country's admin1 units
            a1 = iu.compute_hazard_index(a1, hazs_max_col="hazs_max", output_col="cindex")
            for pc in POP:
                a1 = iu.compute_exposure_index(a1, f"{pc}_log_exposure", f"{pc}_relative_exposure", output_col=f"cexpi_{pc}")
                n_units = len(a1)
                if n_units < 2:
                    # A single admin1 unit has no peers to rank against, so a
                    # country-scope class is undefined. The global classification
                    # leaves these empty (NaN) rather than 0 -- 0 already means
                    # "no exposure" (aexp == 0) -- so match that convention here.
                    a1[f"cexpc_{pc}"] = np.nan
                elif a1[f"cexpi_{pc}"].max() == 0:
                    a1[f"cexpc_{pc}"] = 0
                else:
                    k = min(5, max(3, n_units))
                    a1 = iu.compute_exposure_class(a1, f"cexpi_{pc}", f"{pc}_log_exposure", k=k, output_col=f"cexpc_{pc}")

            a1["hazard_code"] = code
            a1["hazard_short"] = short_code.get(code, code)
            long_rows.append(a1)
            print(f"  {code}: {len(a1)} admin1 units")

        if not long_rows:
            # Every admin2 row for this country is flagged (Insufficient Data /
            # Not Applicable), so aggregate_counts drops them all and there is
            # nothing to summarize. Report it plainly instead of letting
            # pd.concat raise "No objects to concatenate".
            raise RuntimeError("no usable admin2 rows (all flagged) for any hazard")

        long = pd.concat(long_rows, ignore_index=True)

        # `n_children` (from aggregation_functions.aggregate_counts) counts the admin2
        # units aggregated into each admin1 -- NOT the under-18 population. Renamed to
        # `n_admin2` to remove that ambiguity in this child-hazard dataset.
        long = long.rename(columns={"n_children": "n_admin2"})

        print(f"\nLONG: {long.shape}, hazards={long.hazard_code.nunique()}, admin1={long.adm1_ucode.nunique()}")

        # ============================================================
        # LONG output (tidy: one row per admin1 x hazard)
        # ============================================================
        rel_cols = ["total_relative_exposure", "under_18_total_relative_exposure"]
        cexpi_cols = [f"cexpi_{pc}" for pc in POP]
        cexpc_cols = [f"cexpc_{pc}" for pc in POP]
        # hazard_short is dropped from the long output (hazard_code is the full identifier);
        # it is still used internally below to build the WIDE pivot columns.
        long_cols = (
            ["hazard_code", "iso3", "adm0_ucode", "adm1_ucode", "adm1_name",
             "unicef_region", "n_admin2", "hazs_mean", "hazs_std", "hazs_min", "hazs_max"]
            + agg.COUNT_COLS + rel_cols + ["cindex"] + cexpi_cols + cexpc_cols + ["status"]
        )
        # Coverage diagnostics travel with the long output so a consumer can
        # judge whether hazs_mean/std/min/max are based on enough of the unit.
        # Covered/total is TWO numeric columns, never a "3/7" string: Excel
        # autoconverts that to a date on open ("5/12" -> 05-Dec).
        # Coverage as two numeric counts, never a "3/7" string (Excel turns
        # that into a date: "5/12" -> 05-Dec). n_admin2_covered is inserted
        # directly after n_admin2 so the pair is always read together.
        if "n_admin2_covered" in long.columns and "n_admin2_covered" not in long_cols:
            if "n_admin2" in long_cols:
                long_cols.insert(long_cols.index("n_admin2") + 1, "n_admin2_covered")
            else:
                long_cols = long_cols + ["n_admin2_covered"]
        long_out = long[[c for c in long_cols if c in long.columns]].copy()
        # Explicit, stable ordering: hazard then adm1_ucode (pcode). The admin1
        # sequence within each hazard block then matches both workbooks exactly.
        long_out = long_out.sort_values(["hazard_code", "adm1_ucode"])
        long_out = iu.round_cols(long_out)
        long_path = f"{out_dir}/{COUNTRY_UCODE}_admin1_long_{PROD_DATE}.csv"
        long_out.to_csv(long_path, index=False)
        print(f"wrote {long_path}  ({long_out.shape})")

        # ============================================================
        # WIDE output (one row per admin1; per-hazard column blocks) for prioritization
        # ============================================================
        wide = long.pivot_table(
            index=["adm1_ucode", "adm1_name"],
            columns="hazard_short",
            values=WIDE_VALUES,
            aggfunc="first",
        )
        wide.columns = [f"{sc}_{val}" for val, sc in wide.columns]
        wide = wide.reset_index()

        # Cross-hazard prioritization helpers: how many hazards at each class, worst class, mean index
        class_cols = [c for c in wide.columns if c.endswith("_cexpc_under_18_total")]
        index_cols = [c for c in wide.columns if c.endswith("_cexpi_under_18_total")]
        wide["max_cexpc"] = wide[class_cols].max(axis=1)
        wide["n_hazards_class5"] = (wide[class_cols] == 5).sum(axis=1)
        wide["n_hazards_class4plus"] = (wide[class_cols] >= 4).sum(axis=1)
        wide["mean_cexpi"] = wide[index_cols].mean(axis=1).round(1)
        # Ordered by adm1_ucode (pcode), NOT by priority: COs read the long CSV
        # and both workbooks side by side, so every output must present the
        # admin1 units in the same order. The priority columns (max_cexpc,
        # n_hazards_class5, n_hazards_class4plus, mean_cexpi) are still in the
        # sheet, so a reader can re-sort on them in Excel whenever they want
        # the ranked view.
        wide = wide.sort_values("adm1_ucode")

        # The wide table is not written as CSV -- it is delivered as the colour-coded
        # Excel files built in the cells below, which use this same `wide` DataFrame.
        print(f"wide table built in memory  ({wide.shape})")

        # ============================================================
        # WIDE output as a COLOR-CODED Excel (prioritization view)
        # Columns are grouped BY HAZARD (Class | Exposure Index | Hazard Index), with a
        # merged two-row header: hazard display name on top, variable label below.
        # The exposure-CLASS column of each hazard is filled with the official CCRI
        # palette (config/class_palette.json); text flips to white on dark fills.
        # Labels come from hazard_info.json `display_name` and config/column_labels.json.
        # ============================================================
        import json
        from openpyxl import Workbook
        from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

        PALETTE = json.loads((CONFIG_DIR / "class_palette.json").read_text(encoding="utf-8"))
        PAL_CATS, CAT_ALIAS, CLASS_LABELS = PALETTE["categories"], PALETTE["hazard_info_category_alias"], PALETTE["class_labels"]
        LABELS = json.loads((CONFIG_DIR / "column_labels.json").read_text(encoding="utf-8"))
        VAR_LABELS, SUMMARY_LABELS, ID_LABELS = LABELS["variables"], LABELS["summary_columns"], LABELS["id_columns"]

        short_disp = {r["short_code"]: r["display_name"] for _, r in hazard_info.iterrows()}
        short_cat = {r["short_code"]: CAT_ALIAS.get(r["category"], r["category"]) for _, r in hazard_info.iterrows()}

        # Per-hazard column order within each group. Matches WIDE_VALUES.
        GROUP_ORDER = ["cexpc_under_18_total", "cexpi_under_18_total", "cindex"]  # Children: Class, Exposure Index, Hazard Index

        # Hazards present in `wide`, in hazard_info order (then any extras, e.g. conflict full-code).
        present = [r["short_code"] for _, r in hazard_info.iterrows()
                   if any(f"{r['short_code']}_{v}" in wide.columns for v in GROUP_ORDER)]
        known = set(present)
        for col in wide.columns:
            for v in GROUP_ORDER:
                if col.endswith(f"_{v}"):
                    sc = col[:-len(f"_{v}")]
                    if sc not in known:
                        present.append(sc); known.add(sc)

        id_cols = ["adm1_ucode", "adm1_name"]
        summary_cols = [c for c in ["max_cexpc", "n_hazards_class5", "n_hazards_class4plus", "mean_cexpi"] if c in wide.columns]
        group_spans = [(sc, [f"{sc}_{v}" for v in GROUP_ORDER if f"{sc}_{v}" in wide.columns]) for sc in present]
        group_spans = [(sc, cols) for sc, cols in group_spans if cols]
        final_cols = id_cols + [c for _, cols in group_spans for c in cols] + summary_cols
        wide2 = wide[final_cols].copy()


        def _class_int(v):
            n = pd.to_numeric(v, errors="coerce")
            if pd.isna(n):
                return None
            n = int(n)
            return n if 1 <= n <= 5 else None


        def _fill(cat, v):
            n = _class_int(v)
            if n is None:
                return None, None
            hx = PAL_CATS.get(cat, {}).get(CLASS_LABELS[n - 1])
            return (PatternFill("solid", fgColor="FF" + hx.lstrip("#").upper()), hx) if hx else (None, None)


        def _white(hx):
            h = hx.lstrip("#"); r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
            return 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5


        wb = Workbook()
        ws = wb.active
        ws.title = "prioritization"
        H1, H2, DATA0 = 1, 2, 3  # header row 1 (hazard), header row 2 (variable), first data row

        thin = Side(style="thin", color="DDDDDD")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)
        hdr_fill = PatternFill("solid", fgColor="FF305496")
        hdr_font = Font(color="FFFFFF", bold=True)
        center = Alignment(horizontal="center", vertical="center", wrap_text=True)

        col = 1
        for name in id_cols:  # id columns: label spans both header rows
            ws.merge_cells(start_row=H1, start_column=col, end_row=H2, end_column=col)
            c = ws.cell(H1, col, ID_LABELS.get(name, name)); c.fill = hdr_fill; c.font = hdr_font; c.alignment = center
            col += 1
        for sc, cols in group_spans:  # hazard groups: merged name on row1, variable on row2
            start = col
            for cn in cols:
                var = cn[len(sc) + 1:]
                c2 = ws.cell(H2, col, VAR_LABELS.get(var, var)); c2.fill = hdr_fill; c2.font = hdr_font; c2.alignment = center; c2.border = border
                col += 1
            ws.merge_cells(start_row=H1, start_column=start, end_row=H1, end_column=col - 1)
            c1 = ws.cell(H1, start, short_disp.get(sc, sc)); c1.fill = hdr_fill; c1.font = hdr_font; c1.alignment = center; c1.border = border
        for name in summary_cols:
            ws.merge_cells(start_row=H1, start_column=col, end_row=H2, end_column=col)
            c = ws.cell(H1, col, SUMMARY_LABELS.get(name, name)); c.fill = PatternFill("solid", fgColor="FF7F7F7F"); c.font = hdr_font; c.alignment = center
            col += 1

        for i, (_, row) in enumerate(wide2.iterrows()):  # data
            for j, name in enumerate(final_cols, start=1):
                ws.cell(DATA0 + i, j, row[name] if pd.notna(row[name]) else None)

        name_to_col = {name: j for j, name in enumerate(final_cols, start=1)}
        colored = 0
        for sc, _ in group_spans:  # color each hazard's class column
            cat = short_cat.get(sc); ccol = name_to_col.get(f"{sc}_cexpc_under_18_total")
            if cat is None or ccol is None:
                continue
            for i in range(len(wide2)):
                cell = ws.cell(DATA0 + i, ccol)
                fill, hx = _fill(cat, cell.value)
                if fill is None:
                    continue
                cell.fill = fill
                cell.font = Font(color="FFFFFF", bold=True) if _white(hx) else Font(bold=True)
                cell.alignment = Alignment(horizontal="center"); cell.border = border
                colored += 1

        ws.freeze_panes = "C3"  # below the 2-row header, right of the id columns
        xlsx_path = f"{out_dir}/{COUNTRY_UCODE}_admin1_wide_hazard_exposure_{PROD_DATE}.xlsx"
        wb.save(xlsx_path)
        print(f"wrote {xlsx_path}  ({len(group_spans)} hazard groups, {colored} class cells colored)")

        # ============================================================
        # EXPOSURE-focused wide Excel (second prioritization view)
        # Focuses on WHO is exposed rather than hazard intensity. Layout:
        #   front:  Admin1 Code | Admin1 Name | Total Population | Child Population
        #   per hazard (4 cols): Class (Total) | Class (Children) | Exposed (Total) | Exposed (Children)
        # Both class columns are colored with the CCRI palette. Population/exposed counts
        # use thousands formatting. Front population is the FULL admin1 total (max of
        # pop_total across hazards == the unmasked global-CSV admin1 population).
        # ============================================================
        from openpyxl import Workbook

        # Configs (reload so this cell is self-contained regardless of run order)
        _PAL = json.loads((CONFIG_DIR / "class_palette.json").read_text(encoding="utf-8"))
        PAL_CATS, CAT_ALIAS, CLASS_LABELS = _PAL["categories"], _PAL["hazard_info_category_alias"], _PAL["class_labels"]
        _LAB = json.loads((CONFIG_DIR / "column_labels.json").read_text(encoding="utf-8"))
        EXP_VARS, FRONT_LAB, ID_LAB = _LAB["exposure_view_variables"], _LAB["exposure_view_front_columns"], _LAB["id_columns"]

        code_disp = {r["code"]: r["display_name"] for _, r in hazard_info.iterrows()}
        code_cat = {r["code"]: CAT_ALIAS.get(r["category"], r["category"]) for _, r in hazard_info.iterrows()}

        EXP_VALUES = ["cexpc_total", "cexpc_under_18_total", "aexp", "u18_aexp"]
        CLASS_COLS = {"cexpc_total", "cexpc_under_18_total"}

        # hazards present in `long`, in hazard_info order (retired ones already filtered upstream)
        present = [c for c in hazard_info["code"] if c in set(long["hazard_code"])]

        # Full admin1 population = max of pop_total across hazards (== unmasked total).
        front = (long.groupby(["adm1_ucode", "adm1_name"], as_index=False)
                     .agg(pop_total=("pop_total", "max"), u18_pop_total=("u18_pop_total", "max")))

        # Pivot each exposure value into admin1 x hazard, then assemble hazard-grouped.
        wide_exp = front.set_index(["adm1_ucode", "adm1_name"])
        col_meta = []  # (final_col_name, hazard_code, value)
        for v in EXP_VALUES:
            _p = long.pivot_table(index=["adm1_ucode", "adm1_name"], columns="hazard_code", values=v, aggfunc="first")
            for code in present:
                if code in _p.columns:
                    wide_exp[f"{code}__{v}"] = _p[code]
        # reorder columns hazard-grouped
        for code in present:
            for v in EXP_VALUES:
                n = f"{code}__{v}"
                if n in wide_exp.columns:
                    col_meta.append((n, code, v))
        wide_exp = wide_exp[[n for n, _, _ in col_meta]].reset_index()
        wide_exp = front.merge(wide_exp, on=["adm1_ucode", "adm1_name"])

        # sort hotspots on top: worst total-population class, then total exposed
        class_total_cols = [n for (n, c, v) in col_meta if v == "cexpc_total"]
        # Same pcode ordering as the long CSV and the prioritization workbook.
        wide_exp = wide_exp.sort_values("adm1_ucode")


        def _class_int(v):
            n = pd.to_numeric(v, errors="coerce")
            if pd.isna(n):
                return None
            n = int(n)
            return n if 1 <= n <= 5 else None


        def _fill(cat, v):
            n = _class_int(v)
            if n is None:
                return None, None
            hx = PAL_CATS.get(cat, {}).get(CLASS_LABELS[n - 1])
            return (PatternFill("solid", fgColor="FF" + hx.lstrip("#").upper()), hx) if hx else (None, None)


        def _white(hx):
            h = hx.lstrip("#"); r, g, b = (int(h[i:i + 2], 16) / 255 for i in (0, 2, 4))
            return 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5


        wb = Workbook()
        ws = wb.active
        ws.title = "exposure"
        H1, H2, DATA0 = 1, 2, 3

        hdr_fill = PatternFill("solid", fgColor="FF305496")
        front_fill = PatternFill("solid", fgColor="FF548235")
        hdr_font = Font(color="FFFFFF", bold=True)
        center = Alignment(horizontal="center", vertical="center", wrap_text=True)
        thin = Side(style="thin", color="DDDDDD")
        border = Border(left=thin, right=thin, top=thin, bottom=thin)

        id_and_front = ["adm1_ucode", "adm1_name", "pop_total", "u18_pop_total"]
        final_cols = id_and_front + [n for (n, _, _) in col_meta]

        col = 1
        for name in id_and_front:
            ws.merge_cells(start_row=H1, start_column=col, end_row=H2, end_column=col)
            c = ws.cell(H1, col, ID_LAB.get(name) or FRONT_LAB.get(name) or name)
            c.fill = front_fill if name in FRONT_LAB else hdr_fill
            c.font = hdr_font; c.alignment = center
            col += 1
        for code in present:
            group = [(n, v) for (n, c, v) in col_meta if c == code]
            if not group:
                continue
            start = col
            for n, v in group:
                c2 = ws.cell(H2, col, EXP_VARS.get(v, v)); c2.fill = hdr_fill; c2.font = hdr_font; c2.alignment = center; c2.border = border
                col += 1
            ws.merge_cells(start_row=H1, start_column=start, end_row=H1, end_column=col - 1)
            c1 = ws.cell(H1, start, code_disp.get(code, code)); c1.fill = hdr_fill; c1.font = hdr_font; c1.alignment = center; c1.border = border

        for i, (_, row) in enumerate(wide_exp.iterrows()):
            for j, name in enumerate(final_cols, start=1):
                ws.cell(DATA0 + i, j, row[name] if pd.notna(row[name]) else None)

        name_to_col = {name: j for j, name in enumerate(final_cols, start=1)}
        colored = 0
        for (n, code, v) in col_meta:
            if v not in CLASS_COLS:
                continue
            cat = code_cat.get(code)
            if cat is None:
                continue
            ci = name_to_col[n]
            for i in range(len(wide_exp)):
                cell = ws.cell(DATA0 + i, ci)
                fill, hx = _fill(cat, cell.value)
                if fill is None:
                    continue
                cell.fill = fill
                cell.font = Font(color="FFFFFF", bold=True) if _white(hx) else Font(bold=True)
                cell.alignment = Alignment(horizontal="center"); cell.border = border
                colored += 1

        for name, j in name_to_col.items():  # thousands on population + exposed counts
            if name in ("pop_total", "u18_pop_total") or name.endswith("__aexp") or name.endswith("__u18_aexp"):
                for i in range(len(wide_exp)):
                    ws.cell(DATA0 + i, j).number_format = "#,##0"

        ws.freeze_panes = "C3"
        exp_path = f"{out_dir}/{COUNTRY_UCODE}_admin1_exposure_{PROD_DATE}.xlsx"
        wb.save(exp_path)
        print(f"wrote {exp_path}  ({len(present)} hazards, {colored} class cells colored)")

        # ============================================================
        # Verification
        # ============================================================
        # 1. POPULATION INVARIANCE (the check that would have caught the
        #    2026-09 denominator defect). Population must be identical for
        #    every hazard and must reconcile with the admin2 reference, for
        #    ALL SIX population classes -- not just pop_total.
        pop_ref_adm1 = None
        if pop_ref is not None:
            _b = agg.load_hazard_global(codes[0], ADM2_PROD_DATE, SOURCE_FOLDER)
            _b = _b[_b["adm0_ucode"] == COUNTRY_UCODE][["adm2_ucode", "adm1_ucode"]]
            pop_ref_adm1 = agg.apply_population_reference(
                _b.assign(**{c: np.nan for c in agg.COUNT_COLS}), pop_ref)

        problems = agg.validate_population_totals(long, pop_ref_adm1, "adm1_ucode")
        if problems.empty:
            n_cls = sum(1 for _, p in agg.CLASS_TO_COLS.values() if p in long.columns)
            print(f"population: OK -- invariant across {long.hazard_code.nunique()} "
                  f"hazards for all {n_cls} population classes, matches admin2 reference")
        else:
            print(f"population: {len(problems)} DISCREPANCIES")
            print(problems.head(12).to_string(index=False))
            raise AssertionError(
                f"{COUNTRY_UCODE}: population is not hazard-invariant "
                f"({len(problems)} discrepancies) -- refusing to publish")

        # 2. Conservation: summed admin1 aexp == summed admin2 aexp per hazard.
        #    Compared over ALL rows (not status=="") because No Data units now
        #    legitimately contribute 0 exposure rather than being dropped.
        for code in codes[:3]:
            df = agg.load_hazard_global(code, ADM2_PROD_DATE, SOURCE_FOLDER)
            c = df[(df["adm0_ucode"] == COUNTRY_UCODE)
                   & (~df["adm0_ucode"].isin(agg.POP_BLANK_COUNTRIES))]
            adm2_sum = pd.to_numeric(c["aexp"], errors="coerce").fillna(0).sum()
            a1_sum = long.loc[long.hazard_code == code, "aexp"].sum()
            print(f"{code}: adm2 aexp={adm2_sum:,.0f}  admin1 aexp={a1_sum:,.0f}  "
                  f"match={abs(adm2_sum - a1_sum) < 1}")

        # 3. Index/class ranges
        print("\ncexpi_total in [0,10]:", long["cexpi_total"].between(0, 10).all())
        print("cexpc_total in {0..5}:", set(long["cexpc_total"].dropna().unique()).issubset({0, 1, 2, 3, 4, 5}))
        print("admin1 count == distinct adm1_ucode:", long.adm1_ucode.nunique())

        # 4. Coverage summary: where are the hazard statistics thin?
        if "n_admin2_covered" in long.columns:
            thin = long[(long.n_admin2_covered < 0.5 * long.n_admin2)
                        & long.hazs_mean.notna()]
            print(f"thin-coverage rows (<half the admin2 units have data, "
                  f"hazs_mean still reported): {len(thin)} of {len(long)}")

        summary_rows.append({'adm0_ucode': COUNTRY_UCODE,
                             'admin1_units': int(long.adm1_ucode.nunique()),
                             'hazards': int(long.hazard_code.nunique()),
                             'status': 'ok'})
    except Exception as _e:
        print(f'  ERROR {type(_e).__name__}: {_e}')
        summary_rows.append({'adm0_ucode': COUNTRY_UCODE, 'admin1_units': 0,
                             'hazards': 0, 'status': f'{type(_e).__name__}: {_e}'})

run_summary = pd.DataFrame(summary_rows)
print()
print('ok=', (run_summary.status == 'ok').sum(),
      ' failed=', (run_summary.status != 'ok').sum())
if (run_summary.status != 'ok').any():
    print(run_summary[run_summary.status != 'ok'].to_string(index=False))
